# Live Prediction
Real-time ML prediction pipeline that reads sensor data, classifies activity state, and triggers alerts.

**Requires:** ESP32 connected via USB (COM4) and trained model file `rf_activity_model.pkl`.

In [1]:
!pip install pyserial
import serial
import time
import joblib
import numpy as np
import pandas as pd
import json
from pathlib import Path

In [2]:
# serial port, timing and alert threshold settings
PORT = "COM4"
BAUD = 115200

WINDOW_SIZE = 10
SAMPLE_INTERVAL = 0.2

PREDICTION_WINDOW = 5
ACTIVE_CONFIRM_TIME = 5
NOT_PRESENT_TIMEOUT = 2.0
NOT_PRESENT_DISTANCE = 90       # cm — anything beyond this is "not present"

BEEP_TIME = 30
TAP_TIME = 60

STATE_FILE = Path("state.json")
COMMANDS_FILE = Path("commands.json")

In [3]:
# load the trained random forest model from disk
model = joblib.load("rf_activity_model.pkl")

# open serial connection to the ESP32
ser = serial.Serial(PORT, BAUD, timeout=1)
time.sleep(2)

In [4]:
# rolling buffers for sensor readings and predictions
buffer = []
prediction_buffer = []

last_valid_time = time.time()
sedentary_start_time = None
active_start_time = None

beep_sent = False
tap_sent = False

In [5]:
# helpers to read and write state and command JSON files
def load_state():
    try:
        with open(STATE_FILE, "r") as f:
            return json.load(f)
    except Exception:
        return {
            "activity": "unknown",
            "distance": None,
            "inactive_time": 0,
            "last_update": 0
        }

def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

def load_commands():
    try:
        with open(COMMANDS_FILE, "r") as f:
            return json.load(f)
    except Exception:
        return {
            "manual_beep": False,
            "manual_tap": False,
            "manual_rest": False,
            "mute": False,
            "tap_enabled": True,
            "system_enabled": True
        }

def clear_commands(cmds):
    cmds["manual_beep"] = False
    cmds["manual_tap"] = False
    cmds["manual_rest"] = False
    with open(COMMANDS_FILE, "w") as f:
        json.dump(cmds, f, indent=2)

## Main Loop
**Note:** This cell runs continuously. Use the stop/interrupt button to end it.

In [6]:
print("=== SitLess | Live ML + Dashboard Integration ===")

while True:
    state = load_state()
    cmds = load_commands()

    # check for manual commands sent from the dashboard
    if cmds.get("manual_beep") and not cmds.get("mute", False):
        ser.write(b"BEEP\n")

    if cmds.get("manual_tap") and cmds.get("tap_enabled", True):
        ser.write(b"TAP\n")

    if cmds.get("manual_rest"):
        ser.write(b"REST\n")

    clear_commands(cmds)

    # skip everything if the system has been turned off
    if not cmds.get("system_enabled", True):
        state["last_update"] = time.time()
        save_state(state)
        time.sleep(0.1)
        continue

    # read the latest distance line from the serial port
    line = ser.readline().decode(errors="ignore").strip()
    if not line.startswith("DIST:"):
        state["last_update"] = time.time()
        save_state(state)
        time.sleep(0.05)
        continue

    try:
        dist = int(line.split(":")[1])
        state["distance"] = dist

        # if reading is out of range, mark user as not present
        if dist < 0 or dist > NOT_PRESENT_DISTANCE:
            if time.time() - last_valid_time > NOT_PRESENT_TIMEOUT:
                buffer.clear()
                prediction_buffer.clear()
                sedentary_start_time = None
                active_start_time = None
                beep_sent = False
                tap_sent = False
                state["activity"] = "not_present"
                state["inactive_time"] = 0

            state["last_update"] = time.time()
            save_state(state)
            time.sleep(0.05)
            continue

        last_valid_time = time.time()
        buffer.append(dist)

        if len(buffer) < WINDOW_SIZE:
            state["last_update"] = time.time()
            save_state(state)
            time.sleep(0.05)
            continue

        # extract features from the last 10 distance readings
        data = np.array(buffer[-WINDOW_SIZE:])
        features = pd.DataFrame([[
            np.mean(data),
            np.std(data),
            np.max(data) - np.min(data),
            np.mean(np.abs(np.diff(data))),
            WINDOW_SIZE * SAMPLE_INTERVAL
        ]], columns=[
            "mean_distance",
            "std_distance",
            "range_distance",
            "rate_of_change",
            "presence_duration"
        ])

        # run the classifier and take majority vote over last 5 predictions
        prediction = model.predict(features)[0]
        prediction_buffer.append(prediction)
        prediction_buffer = prediction_buffer[-PREDICTION_WINDOW:]

        stable_prediction = max(set(prediction_buffer), key=prediction_buffer.count)
        state["activity"] = stable_prediction

        now = time.time()

        # reset inactivity tracking once movement is confirmed
        if stable_prediction == "active":
            if active_start_time is None:
                active_start_time = now

            if now - active_start_time >= ACTIVE_CONFIRM_TIME:
                sedentary_start_time = None
                beep_sent = False
                tap_sent = False
                state["inactive_time"] = 0
        else:
            active_start_time = None

        # track sitting duration and trigger alerts if needed
        if stable_prediction == "sedentary":
            if sedentary_start_time is None:
                sedentary_start_time = now

            inactive_time = int(now - sedentary_start_time)
            state["inactive_time"] = inactive_time

            if inactive_time >= BEEP_TIME and not beep_sent and not cmds.get("mute", False):
                ser.write(b"BEEP\n")
                beep_sent = True

            # trigger the arm tap after 60 seconds of sitting
            if (
                inactive_time >= TAP_TIME
                and not tap_sent
                and cmds.get("tap_enabled", True)
            ):
                ser.write(b"TAP\n")
                tap_sent = True

        buffer = buffer[-WINDOW_SIZE:]

        state["last_update"] = time.time()
        save_state(state)
        time.sleep(0.05)

    except ValueError:
        state["last_update"] = time.time()
        save_state(state)
        time.sleep(0.05)

=== SitLess | Live ML + Dashboard Integration ===


KeyboardInterrupt: 

Note: The KeyboardInterrupt above is expected this script runs continuously in an infinite loop to monitor sensor data in real time. It must be manually stopped when no longer needed.